# Challenge-Response and Replay

## Goal

This notebook shows why sending a password, or even the same password hash, is replayable. Then it
shows the challenge-response idea: the client sends a value computed from a fresh nonce and a
password-derived secret, so the network message is different at every login.

The code is intentionally small so the mechanics are visible. Real remote authentication protocols
must also handle TLS, phishing, server authentication, device compromise, key management, logging,
and recovery.

In [ ]:
import hashlib
import hmac
import os


# Shared secret for a toy clinician account.
username = "dr.moretti"
password = "blue-river-calm-ward"


def h(data: bytes) -> bytes:
    # A compact helper for SHA-256.
    return hashlib.sha256(data).digest()


# The client can derive this value after the user types the password.
client_password_hash = h(password.encode("utf-8"))

# The hospital server stores the corresponding verifier, not the plaintext password.
stored_password_hash = client_password_hash

print(username, "stored verifier:", stored_password_hash.hex()[:32] + "...")

## Static hash: replayable

If the client sends the same password hash every time, that hash becomes a 	extbf{password
equivalent}. An attacker can capture it once and replay it without knowing the original password.

In [ ]:
def static_hash_login_message(password: str) -> bytes:
    # This message is always the same for the same password.
    # It avoids sending plaintext, but it is still replayable.
    return h(password.encode("utf-8"))


captured_static_hash = static_hash_login_message(password)
print("Captured static hash:", captured_static_hash.hex()[:32] + "...")

# The attacker replays exactly the same bytes later.
replay_accepted = hmac.compare_digest(captured_static_hash, stored_password_hash)
print("Replay accepted?", replay_accepted)

## Challenge-response with a nonce

The server sends a fresh random nonce. The client computes a response over the nonce and the
password-derived value. The client does 	extbf{not} send the password and does 	extbf{not} send
the raw password hash. It sends a response that changes when the nonce changes.

In [ ]:
def make_challenge() -> bytes:
    # A nonce should be unpredictable and should not be reused.
    return os.urandom(16)


def response_for_nonce(nonce: bytes, password_hash: bytes) -> bytes:
    # HMAC binds the response to this nonce and this password-derived value.
    # Different nonce, different transmitted response.
    return hmac.new(password_hash, nonce, hashlib.sha256).digest()


def server_accepts(nonce: bytes, response: bytes) -> bool:
    # The server recomputes the expected response from its stored verifier.
    expected = response_for_nonce(nonce, stored_password_hash)
    return hmac.compare_digest(response, expected)


nonce_1 = make_challenge()
response_1 = response_for_nonce(nonce_1, client_password_hash)

print("First login accepted?", server_accepts(nonce_1, response_1))

In [ ]:
# A replay against a new nonce should fail.
nonce_2 = make_challenge()
print("Replay old response with new nonce accepted?", server_accepts(nonce_2, response_1))

# A fresh response for the new nonce should succeed.
response_2 = response_for_nonce(nonce_2, client_password_hash)
print("Fresh response accepted?", server_accepts(nonce_2, response_2))

print("Response 1:", response_1.hex()[:32] + "...")
print("Response 2:", response_2.hex()[:32] + "...")
print("Same password-derived value, different nonce, different response:", response_1 != response_2)

## Hospital interpretation

For St. Isidore remote EHR access, the important concept is freshness. The response must prove
knowledge of the password-derived secret for this login attempt, not for a login observed
yesterday. The password-derived value may be stable, but the transmitted response changes because
the nonce changes.